## Libraries and Initial Setup

This section imports the Python libraries required for the analysis, including packages for data manipulation, statistical analysis, visualisation, linear regression, and neural networks. Display and plotting settings are also configured, before loading and displaying the Finance 384 stock data dictionary.


In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import plot_model

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
# Initialize random seeds for reproducibility

dictionary = pd.read_csv('FINANCE384_stock_month_data_dictionary.csv')
display(dictionary)

,variable,description,role
0,date,End of month t (CRSP month-end trading date),identifier; not a predictor
1,permno,CRSP permanent security identifier,identifier; not a predictor
2,ticker,Exchange ticker at month t,identifier; not a predictor
3,comnam,Company name at month t,identifier; not a predictor
4,industry,Coarse SIC division label (Agriculture collaps...,"classification, known at t; permitted as a pre..."
5,ff49_code,Fama-French 49-industry code (Ships collapsed ...,"classification, known at t; permitted as a pre..."
6,ff49_name,Fama-French 49-industry name (Ships collapsed ...,"classification, known at t; permitted as a pre..."
7,gvkey,Compustat firm identifier (missing if no CCM l...,identifier; not a predictor
8,size,log market capitalization (USD) at end of mont...,"characteristic, contemporaneous with t"
9,bm,Book equity (6-month availability lag) / marke...,"characteristic, contemporaneous with t"


## Feature Selection

This section identifies the variables that are eligible to be used as predictors based on their classification in the data dictionary. Contemporaneous and predetermined characteristics are extracted, while classifications known at time \(t\) are also identified. The eligible characteristics are then combined to form the final feature set used in the analysis.


In [42]:
determined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, contemporaneous with t", "variable"].tolist()

predetermined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, predetermined at t-1", "variable"].tolist()

determined_classifications = dictionary.loc[
    dictionary["role"] == "classification, known at t; permitted as a predictor if encoded", "variable"].tolist()

print(f"Rows: {len(dictionary):,}")
print(f"Determined characteristics: {determined_characteristics}")
print(f"Predetermined characteristics: {predetermined_characteristics}")
print(f"Determined classifications: {determined_classifications}")

FEATURES = determined_characteristics + predetermined_characteristics
print("Number of features:", len(FEATURES))

Rows: 27
Determined characteristics: ['size', 'bm', 'vol12', 'beta60', 'ivol60', 'turnover', 'dollar_volume', 'amihud_illiq', 'divyield', 'mkt_12m', 'mkt_vol_12m', 'down_market']
Predetermined characteristics: ['mom12_2', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']
Determined classifications: ['industry', 'ff49_code', 'ff49_name']
Number of features: 18


## Panel Data Loading and Validation

This section loads the development panel and converts the date variable into a datetime format. Basic characteristics of the dataset are then examined, including the number of observations, unique stocks, and months covered. The data is also checked for duplicate stock-date observations and for variation in the number of stocks and months represented across the panel.


In [43]:
panel = pd.read_csv('FINANCE384_assignmentA_development_panel.csv', parse_dates=['date'])

print(f"Rows           : {len(panel):,}")
print(f"Distinct stocks: {panel['permno'].nunique():,}")
print(f"Months         : {panel['date'].nunique():,} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
panel.head()


print(f"Duplicate (permno, date) rows: {panel.duplicated(['permno', 'date']).sum()}")

# Check 2: how many stocks are in each month, and how many months does a stock survive?
stocks_per_month = panel.groupby("date")["permno"].nunique()
months_per_stock = panel.groupby("permno")["date"].nunique()
print(f"Stocks per month : {stocks_per_month.min()} to {stocks_per_month.max()} "
      f"(mean {stocks_per_month.mean():.0f})")
print(f"Months per stock : {months_per_stock.min()} to {months_per_stock.max()} "
      f"(median {months_per_stock.median():.0f})")


Rows           : 198,298
Distinct stocks: 1,252
Months         : 396 (1990-01 to 2022-12)
Duplicate (permno, date) rows: 0
Stocks per month : 497 to 505 (mean 501)
Months per stock : 1 to 396 (median 122)


## Target Variable Construction

This section constructs the one-month-ahead excess return target variable, \(target\_ret\_excess\_tp1\), by shifting each stock's excess return forward by one month. The target is then merged back into the panel using the stock identifier and month.

Several diagnostic checks are performed to identify observations without a valid future return, distinguish stocks reaching the end of their available data from gaps in coverage, and assess missing values in the predictors and target. Summary statistics are also calculated for the selected features and target variable.


In [44]:
TARGET = "target_ret_excess_tp1"

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[["permno", "month", "ret_excess_t"]]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(next_month_return, on=["permno", "month"], how="left", validate="one_to_one")

final_month = (
    panel.groupby("permno")["month"]
    .transform("max")
    .eq(panel["month"])
)

no_target = panel[TARGET].isna()
# Diagnostics
print(f"Rows: {len(panel):,}")
print(f"Usable targets: {panel[TARGET].notna().sum():,}")
print(f"No month t+1 in data: {no_target.sum():,}")
print(f"Stock's last month: {(no_target & final_month).sum():,}")
print(f"Gap in coverage: {(no_target & ~final_month).sum():,}")

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())

print("\nFeature summary:")
display(
    panel[FEATURES + [TARGET]]
    .describe()
    .loc[["mean", "std", "min", "max"]]
)

panel[["permno", "month", "ret_excess_t", TARGET]].head()

Rows: 198,298
Usable targets: 197,016
No month t+1 in data: 1,282
Stock's last month: 1,252
Gap in coverage: 30

Missing values:
size                         0
bm                        4242
vol12                     5268
beta60                   17211
ivol60                   17211
turnover                     0
dollar_volume                0
amihud_illiq                 0
divyield                  5927
mkt_12m                      0
mkt_vol_12m                  0
down_market                  0
mom12_2                   6023
gross_profit               515
roe                       4259
asset_growth               737
leverage                   467
accruals                  6615
target_ret_excess_tp1     1282
dtype: int64

Feature summary:


,size,bm,vol12,beta60,ivol60,turnover,dollar_volume,amihud_illiq,divyield,mkt_12m,mkt_vol_12m,down_market,mom12_2,gross_profit,roe,asset_growth,leverage,accruals,target_ret_excess_tp1
mean,23.0467,0.5098,0.0847,1.0445,0.0725,0.1647,3.3957e+09,0.0003,0.0207,0.0891,0.0404,0.2297,0.1158,0.2994,0.1733,0.1099,0.2625,-0.0482,0.0083
std,1.3306,0.4449,0.0452,0.5579,0.0304,0.1518,6.9844e+09,0.0018,0.0196,0.1661,0.0164,0.4207,0.3274,0.2254,0.2518,0.2472,0.1713,0.0555,0.0990
min,17.3590,0.0082,0.0193,-0.3211,0.0257,0.0104,9.6556e+05,0.0000,0.0000,-0.4330,0.0108,0.0000,-0.9108,-0.1985,-0.8534,-0.4729,0.0000,-0.6748,-0.8703
max,27.5092,11.2483,0.5192,4.0973,0.2498,2.1366,2.3617e+11,0.1307,0.5481,0.6466,0.0894,1.0000,4.2891,1.1964,4.5006,3.0477,1.0184,0.1612,2.6266


,permno,month,ret_excess_t,target_ret_excess_tp1
0,10057,1990-01,-0.0540,-0.0179
1,10057,1990-02,-0.0179,-0.0558
2,10057,1990-03,-0.0558,-0.0485
3,10057,1990-04,-0.0485,-0.0205
4,10057,1990-05,-0.0205,-0.1035


## Risk Target Transformation

This section creates a transformed target variable, \(y_{risk}\), from the one-month-ahead excess return. The transformation uses the natural logarithm of one plus the squared return, providing a measure related to the magnitude of future return movements while reducing the influence of extreme values.

The assertions verify that the transformation does not introduce additional missing values or non-finite observations.


In [45]:
TARGET_A = TARGET
TARGET = "y_risk"
panel[TARGET] = np.log1p(panel[TARGET_A] ** 2)
assert panel[TARGET].notna().equals(panel[TARGET_A].notna())
assert np.isfinite(panel.loc[panel[TARGET].notna(), TARGET]).all()


## Missing Value Treatment

This section handles missing values in the predictor variables by replacing each missing observation with the median value of that feature within the same month. This preserves the cross-sectional structure of the data while avoiding the use of future information.

Observations without a valid target value are then removed, leaving only observations that can be used in the analysis. Finally, the remaining missing values across the selected features and target are checked.


In [46]:
industry = 'ff49_name'

for feature in FEATURES:
    monthly_median = (
        panel.groupby('date')[feature]
        .transform('median'))
    
    panel[feature] = (
        panel[feature]
        .fillna(monthly_median))

panel = panel.loc[panel[TARGET].notna()].copy()

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())


Missing values:
size             0
bm               0
vol12            0
beta60           0
ivol60           0
turnover         0
dollar_volume    0
amihud_illiq     0
divyield         0
mkt_12m          0
mkt_vol_12m      0
down_market      0
mom12_2          0
gross_profit     0
roe              0
asset_growth     0
leverage         0
accruals         0
y_risk           0
dtype: int64


### Train, Validation, and Test Split

The dataset is divided into three time-based samples to preserve the chronological structure of the financial data:

- **Training:** January 1990 to December 2014
- **Validation:** January 2015 to December 2018
- **Test:** January 2019 to November 2022

The training and validation samples are also combined to create the `train_valid` dataset, which can be used for final model training before evaluating performance on the unseen test set.

The summary table reports the number of unique months, observations, and the start and end dates for each sample.

In [47]:
train = panel.loc[panel['month'].between('1990-01', '2014-12')].copy()
valid = panel.loc[panel['month'].between('2015-01', '2018-12')].copy()
test = panel.loc[panel['month'].between('2019-01', '2022-11')].copy()
train_valid = pd.concat([train, valid]).copy()

split_summary = pd.DataFrame([
    {'sample': name, 'months': block['month'].nunique(), 'rows': len(block),
     'start': str(block['month'].min()), 'end': str(block['month'].max())}
    for name, block in [('Train', train), ('Validation', valid), ('Test', test)]
])
display(split_summary)


,sample,months,rows,start,end
0,Train,300,149334,1990-01,2014-12
1,Validation,48,24051,2015-01,2018-12
2,Test,47,23631,2019-01,2022-11


### Sample Size Consistency Check

The number of observations in the Train, Validation, and Test samples is compared against the corresponding sample sizes from Part A.

The `Match` column verifies whether the row counts are identical between Part A and Part B. An assertion is then used to confirm that all three samples match.

If the assertion passes without an error, the sample splits are consistent across both parts of the assignment.

In [48]:
part_a_rows = {"Train": 149334, "Validation": 24051, "Test": 23631}
part_b_rows = {"Train": len(train), "Validation": len(valid), "Test": len(test)}
sample_check = pd.DataFrame({"Part A rows": part_a_rows, "Part B rows": part_b_rows})
sample_check["Match"] = sample_check["Part A rows"].eq(sample_check["Part B rows"])
display(sample_check)
assert sample_check["Match"].all()


,Part A rows,Part B rows,Match
Train,149334,149334,True
Validation,24051,24051,True
Test,23631,23631,True


### Feature Standardization and OLS Benchmark

The input features are standardized using the **training sample only**. The training means and standard deviations are used to transform the training, validation, test, and combined training-validation samples.

This ensures that information from the validation and test periods does not influence the scaling parameters.

The target variable is kept in its original form. An OLS benchmark model is then fitted using the combined **training and validation samples**. The number of estimated coefficients and the model intercept are reported.

- **Standardization:** fitted using `train` only
- **Training:** standardized using training-sample statistics
- **Validation:** standardized using training-sample statistics
- **Test:** standardized using training-sample statistics
- **Final OLS benchmark:** fitted on `train_valid`

In [49]:
def standardize(block, fitted_on):
    return (
        block[FEATURES]
        - fitted_on[FEATURES].mean()
    ) / fitted_on[FEATURES].std(ddof=0)


X_train = standardize(train, fitted_on=train)
X_valid = standardize(valid, fitted_on=train)
X_test = standardize(test, fitted_on=train)
X_train_valid = standardize(train_valid, fitted_on=train)

y_train, y_valid, y_test = train[TARGET], valid[TARGET], test[TARGET]
y_train_valid = train_valid[TARGET]

ols_benchmark = LinearRegression().fit(
    X_train_valid,
    y_train_valid
)

print("Number of coefficients:", len(ols_benchmark.coef_))
print("Intercept:", ols_benchmark.intercept_)

Number of coefficients: 18
Intercept: 0.009756719789081639


### Neural Network Architecture

A feed-forward neural network is defined as a benchmark model for predicting the target variable.

The network consists of:

- **Input layer:** one input for each feature
- **Hidden layer:** 16 neurons with a ReLU activation function
- **Output layer:** 1 neuron with a linear activation function for continuous prediction
- **Optimizer:** Adam with a learning rate of 0.001
- **Loss function:** Mean Squared Error (MSE)

The model is compiled and its architecture is displayed using `model.summary()`.

In [50]:
def neural_network(input):
    model = Sequential()
    model.add(
        Dense(
            16,
            input_shape=(input,),
            activation="relu"))
    model.add(
        Dense(
            1,
            activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mean_squared_error")
    model.summary()
    return model


### Neural Network Training Function

The `run_network` function trains and evaluates a neural network for a given batch size, random seed, and maximum number of epochs.

To ensure reproducibility, the same random seed is applied before each experiment. A fresh neural network is then created and trained using the training sample, with the validation sample used to monitor performance.

**Early stopping** is implemented by monitoring validation loss. Training stops if validation loss does not improve for 5 consecutive epochs, and the model weights from the epoch with the best validation loss are restored.

The function records the following metrics for each epoch:

- Training MSE and RMSE
- Validation MSE and RMSE
- Batch size
- Random seed
- Epoch number

The epoch with the lowest validation RMSE is identified and stored in `model_results` for comparison across different network configurations.

Training is performed with `shuffle=False` to preserve the ordering of the observations.

In [51]:

def run_network(batch_size, SEED, EPOCHS):

    # Ensure every experiment begins from the same random seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(len(FEATURES))

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=1,
        shuffle=False
    )

    # Create the error table
    error_table = pd.DataFrame({
        "Batch Size": batch_size,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"]
    )

    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"]
    )
    print(f"Batch size: {batch_size}, "
        f"Seed: {SEED}, "
        f"Maximum epochs: {EPOCHS}\n")
    display(error_table.set_index("Epoch"))

    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)



### Comparing Batch Sizes

Two neural network models are trained using different batch sizes while keeping the random seed and maximum number of epochs constant.

- **Batch size:** 32 and 16
- **Random seed:** 200
- **Maximum epochs:** 100

The validation RMSE from the best-performing epoch of each model is recorded and compared. The results are sorted by validation RMSE, allowing the batch size associated with the lower validation error to be identified.

In [52]:
# comparing batch sizes
model_results = []

run_network(32, 200, 100)   
run_network(16, 200, 100)
model_results = pd.DataFrame(model_results)
display(model_results.sort_values(by="Validation RMSE").reset_index(drop=True))

/Users/callumcoyne/Documents/UNI/Year_5/Finance_384/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ (None, 16)             │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

...

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 366us/step - loss: 0.0582 - val_loss: 0.0013
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 357us/step - loss: 0.0015 - val_loss: 2.1393e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 357us/step - loss: 9.5178e-04 - val_loss: 1.8273e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 359us/step - loss: 8.3366e-04 - val_loss: 1.6651e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 359us/step - loss: 7.8258e-04 - val_loss: 1.6041e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 355us/step - loss: 7.6298e-04 - val_loss: 1.5415e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 360us/step - loss: 7.5502e-04 - val_loss: 1.5582e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 424us/step - loss: 7.5247e-04 - val_loss: 1.5817e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 384us/step - loss: 7.5015e-04 - val_loss: 1.5605e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 387us/step - loss: 7.5261e-04 - val_loss: 1.5440e-04
Epoch 11/100
9334/933

,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,
1,16,200,0.0582,0.0013,0.2412,0.0364
2,16,200,0.0015,0.0002,0.0384,0.0146
3,16,200,0.0010,0.0002,0.0309,0.0135
4,16,200,0.0008,0.0002,0.0289,0.0129
5,16,200,0.0008,0.0002,0.0280,0.0127
6,16,200,0.0008,0.0002,0.0276,0.0124
7,16,200,0.0008,0.0002,0.0275,0.0125
8,16,200,0.0008,0.0002,0.0274,0.0126
9,16,200,0.0008,0.0002,0.0274,0.0125


,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,16.0,200.0,6.0,0.0008,0.0002,0.0276,0.0124
1,32.0,200.0,18.0,0.0007,0.0002,0.0272,0.0124


### Neural Network Hyperparameter Tuning

The neural network training procedure is extended to allow different **hidden-layer architectures** and **learning rates** to be tested while keeping the batch size, random seed, and maximum number of epochs fixed.

The `neural_network` function constructs a feed-forward neural network with:

- A configurable number of neurons in each hidden layer
- ReLU activation functions for the hidden layers
- One linear output neuron for predicting the continuous target
- Adam optimization with a configurable learning rate
- Mean Squared Error (MSE) as the loss function

The `run_network` function trains each candidate model using:

- **Batch size:** 16
- **Random seed:** 200
- **Maximum epochs:** 100
- **Early stopping:** patience of 5 epochs based on validation loss
- **Shuffle:** disabled to preserve the ordering of the observations

For each completed epoch, training and validation MSE and RMSE are recorded. The model configuration and epoch with the **lowest validation RMSE** are then stored in `model_results`.

Using the same seed and batch size for each candidate ensures that differences in performance are primarily associated with the tested network architecture and learning rate.

In [53]:
EPOCHS = 100
BATCH_SIZE = 16
SEED = 200

def neural_network(input, hidden_layers, learning_rate):
    model = Sequential()

    # First hidden layer
    model.add(
        Dense(
            hidden_layers[0],
            input_shape=(input,),
            activation="relu"))

    # Any additional hidden layers
    for neurons in hidden_layers[1:]:
        model.add(
            Dense(
                neurons,
                activation="relu"))

    # One predicted return
    model.add(
        Dense(
            1,
            activation="linear"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mean_squared_error")
    return model


def run_network(hidden_layers, learning_rate):

    # Give every candidate the same starting seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(
        len(FEATURES),
        hidden_layers,
        learning_rate)

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True)
        ],
        verbose=1,
        shuffle=False)

    # Store the error from every completed epoch
    error_table = pd.DataFrame({
        "Hidden Layers": str(hidden_layers),
        "Learning Rate": learning_rate,
        "Batch Size": BATCH_SIZE,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"])
    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"])
            
    # Display the complete table
    display(
        error_table
        .set_index("Epoch")
        .round(6))


    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)

    return model, history, error_table


### Comparing Neural Network Architectures and Learning Rates

Multiple neural network configurations are evaluated to identify the combination of **hidden-layer architecture** and **learning rate** that provides the lowest validation RMSE.

The following configurations are tested:

- **[32, 16, 8]** with learning rates of 0.0015, 0.001, and 0.0005
- **[16, 8]** with learning rates of 0.0015, 0.001, and 0.0005
- **[32]** with a learning rate of 0.001
- **[16]** with a learning rate of 0.001

For every configuration, the **batch size is fixed at 16**, the **random seed is fixed at 200**, and the maximum number of epochs is 100. Early stopping is used to prevent unnecessary training when validation performance stops improving.

The results from each experiment are combined into `model_results` and sorted by validation RMSE. The configuration with the lowest validation RMSE represents the selected model based on validation performance.

In [54]:
model_results = []
run_network([32, 16, 8], 0.0015)
run_network([32, 16, 8], 0.001)
run_network([32, 16, 8], 0.0005)

run_network([16, 8], 0.0015)
run_network([16, 8], 0.001)
run_network([16, 8], 0.0005)

run_network([32], 0.001)
run_network([16], 0.001)


model_results = pd.DataFrame(model_results)
display(
    model_results
    .sort_values("Validation RMSE")
    .reset_index(drop=True)
)


Epoch 1/100


/Users/callumcoyne/Documents/UNI/Year_5/Finance_384/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 5s 441us/step - loss: 8.8406e-04 - val_loss: 1.6118e-04
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 441us/step - loss: 7.6982e-04 - val_loss: 1.5581e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 474us/step - loss: 7.5871e-04 - val_loss: 1.5295e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 429us/step - loss: 7.4858e-04 - val_loss: 1.5328e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 415us/step - loss: 7.5194e-04 - val_loss: 1.6168e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 405us/step - loss: 7.4412e-04 - val_loss: 1.5503e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 404us/step - loss: 7.4124e-04 - val_loss: 1.5843e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 410us/step - loss: 7.4347e-04 - val_loss: 1.5942e-04


,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,"[32, 16, 8]",0.0015,16,200,0.0009,0.0002,0.0297,0.0127
2,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0277,0.0125
3,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0275,0.0124
4,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0274,0.0124
5,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0274,0.0127
6,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0273,0.0125
7,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0272,0.0126
8,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0273,0.0126


Epoch 1/100


...

Epoch 1/100


/Users/callumcoyne/Documents/UNI/Year_5/Finance_384/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 383us/step - loss: 0.0582 - val_loss: 0.0013
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 398us/step - loss: 0.0015 - val_loss: 2.1393e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 418us/step - loss: 9.5178e-04 - val_loss: 1.8273e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 371us/step - loss: 8.3366e-04 - val_loss: 1.6651e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 365us/step - loss: 7.8258e-04 - val_loss: 1.6041e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 366us/step - loss: 7.6298e-04 - val_loss: 1.5415e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 366us/step - loss: 7.5502e-04 - val_loss: 1.5582e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 362us/step - loss: 7.5247e-04 - val_loss: 1.5817e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 368us/step - loss: 7.5015e-04 - val_loss: 1.5605e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 361us/step - loss: 7.5261e-04 - val_loss: 1.5440e-04
Epoch 11/100
9334/9334 ━━━━━━━━━━

,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,[16],0.001,16,200,0.0582,0.0013,0.2412,0.0364
2,[16],0.001,16,200,0.0015,0.0002,0.0384,0.0146
3,[16],0.001,16,200,0.0010,0.0002,0.0309,0.0135
4,[16],0.001,16,200,0.0008,0.0002,0.0289,0.0129
5,[16],0.001,16,200,0.0008,0.0002,0.0280,0.0127
6,[16],0.001,16,200,0.0008,0.0002,0.0276,0.0124
7,[16],0.001,16,200,0.0008,0.0002,0.0275,0.0125
8,[16],0.001,16,200,0.0008,0.0002,0.0274,0.0126
9,[16],0.001,16,200,0.0008,0.0002,0.0274,0.0125


,Hidden Layers,Learning Rate,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,[32],0.0010,16,200,13,0.0007,0.0002,0.0273,0.0123
1,"[32, 16, 8]",0.0015,16,200,3,0.0008,0.0002,0.0275,0.0124
2,"[32, 16, 8]",0.0005,16,200,4,0.0007,0.0002,0.0271,0.0124
3,[16],0.0010,16,200,6,0.0008,0.0002,0.0276,0.0124
4,"[16, 8]",0.0010,16,200,3,0.0008,0.0002,0.0278,0.0125
5,"[32, 16, 8]",0.0010,16,200,1,0.0009,0.0002,0.0304,0.0125
6,"[16, 8]",0.0005,16,200,5,0.0008,0.0002,0.0275,0.0125
7,"[16, 8]",0.0015,16,200,3,0.0008,0.0002,0.0278,0.0125


### Final Model Selection and Predictions

The neural network configuration selected from the hyperparameter comparison is retrained using the chosen architecture and learning rate:

- **Hidden layers:** `[32]`
- **Learning rate:** `0.001`
- **Batch size:** 16
- **Random seed:** 200
- **Maximum epochs:** 100
- **Early stopping:** patience of 5 epochs

The selected neural network is used to generate predictions for both the validation and test samples.

The OLS benchmark, which was previously fitted using the combined training and validation data, is used to generate predictions for the test sample.

Two result tables are then created containing the relevant identifiers, the actual target return, and model predictions:

- `valid_results` contains validation observations and neural network predictions.
- `test_results` contains test observations, OLS predictions, and neural network predictions.

These tables provide the inputs for the subsequent evaluation of model performance and portfolio construction.

In [55]:
import ast
selected_hidden_layers = [32]
selected_learning_rate = 0.001
model_results = []

selected_model, selected_history, selected_error_table = run_network(
    selected_hidden_layers,
    selected_learning_rate
)

validation_predictions = selected_model.predict(
    X_valid.values,
    verbose=0
).flatten()


ols_predictions = ols_benchmark.predict(
    X_test.values
)

nn_predictions = selected_model.predict(
    X_test.values,
    verbose=0
).flatten()

valid_results = valid[
    ["date", "month", "permno", TARGET]
].copy()

test_results = test[
    ["date", "month", "permno", TARGET]
].copy()



valid_results["NN prediction"] = validation_predictions
test_results["OLS prediction"] = ols_predictions
test_results["NN prediction"] = nn_predictions



Epoch 1/100


/Users/callumcoyne/Documents/UNI/Year_5/Finance_384/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 384us/step - loss: 0.0936 - val_loss: 0.0024
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 401us/step - loss: 0.0030 - val_loss: 8.2031e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 382us/step - loss: 0.0015 - val_loss: 2.2478e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 368us/step - loss: 0.0013 - val_loss: 1.8262e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 373us/step - loss: 0.0010 - val_loss: 1.6662e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 367us/step - loss: 8.5995e-04 - val_loss: 1.6206e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 379us/step - loss: 7.9945e-04 - val_loss: 1.5340e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 407us/step - loss: 7.5930e-04 - val_loss: 1.5290e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4s 379us/step - loss: 7.5282e-04 - val_loss: 1.5352e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 366us/step - loss: 7.4616e-04 - val_loss: 1.5402e-04
Epoch 11/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 4

,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,[32],0.001,16,200,0.0936,0.0024,0.3059,0.0487
2,[32],0.001,16,200,0.0030,0.0008,0.0549,0.0286
3,[32],0.001,16,200,0.0015,0.0002,0.0381,0.0150
4,[32],0.001,16,200,0.0013,0.0002,0.0356,0.0135
5,[32],0.001,16,200,0.0010,0.0002,0.0322,0.0129
6,[32],0.001,16,200,0.0009,0.0002,0.0293,0.0127
7,[32],0.001,16,200,0.0008,0.0002,0.0283,0.0124
8,[32],0.001,16,200,0.0008,0.0002,0.0276,0.0124
9,[32],0.001,16,200,0.0008,0.0002,0.0274,0.0124


/Users/callumcoyne/Documents/UNI/Year_5/Finance_384/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


*** RMSE

### Model Prediction Error

The predictive performance of the neural network and OLS benchmark is evaluated using **Root Mean Squared Error (RMSE)**.

RMSE measures the typical magnitude of prediction errors, with lower values indicating predictions that are closer to the observed target returns.

Three RMSE measures are calculated:

- **Validation RMSE:** performance of the selected neural network on the validation sample.
- **Neural Network Test RMSE:** out-of-sample performance of the selected neural network on the test sample.
- **OLS Test RMSE:** out-of-sample performance of the OLS benchmark on the test sample.

The test-sample RMSE values allow the neural network's out-of-sample predictive performance to be compared directly with the OLS benchmark.

In [56]:

valid_rmse = np.sqrt(
    mean_squared_error(
        valid_results[TARGET],
        valid_results["NN prediction"]
    )
)



nn_test_rmse = np.sqrt(
    mean_squared_error(
        test_results[TARGET],
        test_results["NN prediction"]
    )
)

ols_test_rmse = np.sqrt(
    mean_squared_error(
        test_results[TARGET],
        test_results["OLS prediction"]
    )
)



### Predictive Performance Summary

Model performance is evaluated using two complementary measures: **pooled RMSE** and **mean monthly Spearman correlation**.

The `mean_monthly_spearman` function calculates the Spearman rank correlation between actual and predicted returns separately for each month and then takes the average across all months. This measures how well each model ranks stocks relative to their realised returns within each month.

The final summary reports:

- **Period:** Validation or Test
- **Model:** Neural Network (NN) or OLS
- **Months:** Number of months included in the sample
- **Stock-month rows:** Number of observations
- **Pooled RMSE:** Overall prediction error across all observations
- **Mean Monthly Spearman:** Average cross-sectional rank correlation across months

The validation results evaluate the selected neural network during model development, while the test results provide the final out-of-sample comparison between the neural network and the OLS benchmark.

In [ ]:
def mean_monthly_spearman(df, actual_col, prediction_col):

    monthly_correlations = (
        df.groupby("month")
        .apply(
            lambda x: spearmanr(
                x[actual_col],
                x[prediction_col]
            ).statistic
        )
    )

    return monthly_correlations.mean()

valid_spearman = mean_monthly_spearman(
    valid_results,
    TARGET,
    "NN prediction"
)

nn_test_spearman = mean_monthly_spearman(
    test_results,
    TARGET,
    "NN prediction"
)

ols_test_spearman = mean_monthly_spearman(
    test_results,
    TARGET,
    "OLS prediction"
)

valid_months = valid_results["month"].nunique()
valid_rows = len(valid_results)


test_months = test_results["month"].nunique()
test_rows = len(test_results)


,Period,Model,Months,Stock-month rows,Pooled RMSE,Mean Monthly Spearman
0,Validation,NN,48,24051,0.0123,0.2182
1,Test,NN,47,23631,0.0244,0.2112
2,Test,OLS,47,23631,0.0240,0.2093


### QLIKE

QLIKE evaluates the accuracy of the model's **variance/risk forecasts**. Lower QLIKE indicates a better variance forecast.

For each stock-month:

\[
QLIKE = \frac{r^2}{h}
- \log\left(\frac{r^2}{h}\right) - 1
\]

where:

- \(r^2\) is the realised variance, calculated from the actual excess return.
- \(h\) is the forecast variance produced by the model.
- A variance floor of \(10^{-4}\) is applied to forecast variance.
- A realised variance floor of \(10^{-8}\) prevents numerical issues when the realised return is zero.

The QLIKE function takes inputs in the form:

\[
y = \log(1+r^2)
\]

and converts them back to variance using:

\[
r^2 = \exp(y)-1
\]

The actual returns and predicted returns are therefore first transformed into \(y=\log(1+r^2)\) before calculating QLIKE.

The evaluation follows three steps:

1. Calculate QLIKE for each stock-month for both the OLS and neural network models.
2. Average QLIKE across stocks within each month.
3. Take the simple average of the monthly QLIKE values to obtain the overall test QLIKE.

The same stock-month observations are used for both models to ensure a fair comparison.

In [ ]:
### QLIKE

H_MIN = 1e-4   # variance floor: 1% monthly volatility
R2_MIN = 1e-8  # realised variance floor


# QLIKE loss per stock-month
# Inputs are y = log(1 + r^2)
def qlike(y_true, y_pred, h_min=H_MIN):
    
    # Convert log(1 + r^2) back to realised variance
    r2 = np.maximum(
        np.expm1(np.asarray(y_true)),
        R2_MIN
    )
    
    # Convert log(1 + forecast variance) back to forecast variance
    h = np.maximum(
        np.expm1(np.asarray(y_pred)),
        h_min
    )
    
    ratio = r2 / h
    
    return ratio - np.log(ratio) - 1.0


# Use the same stock-month rows for both models
qlike_results = test_results[
    ["date", "month", TARGET, "OLS prediction", "NN prediction"]
].dropna().copy()


# Convert actual returns to y = log(1 + r^2)
qlike_results["Actual y"] = np.log(
    1 + qlike_results[TARGET] ** 2
)


# Convert OLS predicted returns to y = log(1 + r^2)
qlike_results["OLS y"] = np.log(
    1 + qlike_results["OLS prediction"] ** 2
)


# Convert NN predicted returns to y = log(1 + r^2)
qlike_results["NN y"] = np.log(
    1 + qlike_results["NN prediction"] ** 2
)


# Calculate QLIKE for each stock-month
qlike_results["OLS QLIKE"] = qlike(
    qlike_results["Actual y"],
    qlike_results["OLS y"]
)

qlike_results["NN QLIKE"] = qlike(
    qlike_results["Actual y"],
    qlike_results["NN y"]
)


# Average QLIKE across stocks within each month
monthly_qlike = (
    qlike_results
    .groupby("month")[["OLS QLIKE", "NN QLIKE"]]
    .mean()
)


# Simple average across months
ols_test_qlike = monthly_qlike["OLS QLIKE"].mean()

nn_test_qlike = monthly_qlike["NN QLIKE"].mean()



OLS Test QLIKE: 4.853516330518434
NN Test QLIKE: 5.950794195633281


### Model Performance Summary

The table below compares the neural network (NN) and pooled OLS benchmark using three evaluation metrics:

- **Pooled RMSE:** Measures the overall accuracy of the predicted risk values. Lower RMSE indicates smaller prediction errors.
- **Mean Monthly Spearman:** Measures how well each model ranks stocks according to their predicted risk. Higher Spearman rank correlation indicates better ranking performance.
- **Mean Monthly QLIKE:** Measures the accuracy of the predicted variance. Lower QLIKE indicates a better variance forecast.

The validation set reports RMSE and Spearman correlation for the NN because it was used for model selection. QLIKE is reported only for the test set, where the final NN is compared directly with the pooled OLS benchmark.

The test results allow the richer NN to be compared with the OLS benchmark using both the accuracy of the risk predictions and their ability to rank stocks by risk.

In [60]:
results_summary = pd.DataFrame({

    "Period": [
        "Validation",
        "Test",
        "Test"
    ],

    "Model": [
        "NN",
        "NN",
        "OLS"
    ],

    "Months": [
        valid_months,
        test_months,
        test_months
    ],

    "Stock-month rows": [
        valid_rows,
        test_rows,
        test_rows
    ],

    "Pooled RMSE": [
        valid_rmse,
        nn_test_rmse,
        ols_test_rmse
    ],

    "Mean Monthly Spearman": [
        valid_spearman,
        nn_test_spearman,
        ols_test_spearman
    ],

    "Mean Monthly QLIKE": [
        np.nan,
        nn_test_qlike,
        ols_test_qlike
    ]

})

display(
    results_summary.round(6)
)

,Period,Model,Months,Stock-month rows,Pooled RMSE,Mean Monthly Spearman,Mean Monthly QLIKE
0,Validation,NN,48,24051,0.0123,0.2182,NaN
1,Test,NN,47,23631,0.0244,0.2112,5.9508
2,Test,OLS,47,23631,0.0240,0.2093,4.8535
